# El dolor de los datos — un fallo silencioso

Vamos a reproducir un incidente de datos que **no lanza ninguna excepción**:
entrenaremos un modelo con datos limpios, luego alguien cambiará las unidades
de una columna sin avisar, y veremos que el modelo sigue sirviendo predicciones
sin que ninguna herramienta estándar se queje.

Al final construiremos un contrato de datos de tres niveles y veremos que
**sí** atrapa el problema.

**Pre-requisito:** `uv pip install pandera` (o `pip install pandera` si usas .venv).

In [1]:
# 1. Generamos un dataset sintetico de transacciones de retail.
# 10_000 filas, categoricas realistas, Revenue coherente con Price * Quantity.
import numpy as np
import pandas as pd

RNG = np.random.default_rng(seed=42)
N = 10_000

categorias = RNG.choice(["A", "B", "C", "D"], size=N, p=[0.4, 0.3, 0.2, 0.1])
paises     = RNG.choice(["UK", "DE", "FR", "ES"], size=N, p=[0.6, 0.2, 0.1, 0.1])
price      = RNG.lognormal(mean=1.5, sigma=0.6, size=N).round(2)  # ~libras esterlinas
quantity   = RNG.integers(1, 20, size=N)
revenue    = (price * quantity).round(2)

datos = pd.DataFrame({
    "Category": categorias,
    "Country":  paises,
    "Price":    price,
    "Quantity": quantity,
    "Revenue":  revenue,
})
print(f"Shape: {datos.shape}")
print(f"Precio medio: £{datos['Price'].mean():.2f}  (mediana £{datos['Price'].median():.2f})")
datos.head()

Shape: (10000, 5)
Precio medio: £5.45  (mediana £4.55)


,Category,Country,Price,Quantity,Revenue
0,C,DE,1.50,18,27.00
1,B,DE,3.54,17,60.18
2,C,UK,1.57,8,12.56
3,B,UK,15.26,16,244.16
4,A,UK,5.09,1,5.09


In [2]:
# 2. Entrenamos un modelo lineal que predice Revenue a partir de Price, Quantity y las categoricas.
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

X = datos[["Category", "Country", "Price", "Quantity"]]
y = datos["Revenue"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Category", "Country"]),
    ], remainder="passthrough")),
    ("modelo", LinearRegression()),
])

pipe.fit(X_tr, y_tr)
mae_base = mean_absolute_error(y_te, pipe.predict(X_te))
print(f"MAE del modelo con datos limpios: £{mae_base:.2f}")

MAE del modelo con datos limpios: £12.21


El MAE es bajo (comparado con la media de Revenue). El modelo captura bien la relacion
`Revenue ≈ Price × Quantity`.

---

## Entra el incidente

Imagina que llega un **lote nuevo** desde el proveedor de datos. Alguien
actualizo el software de captura y, sin aviso, los precios ahora llegan en
**peniques (céntimos de libra)** en vez de libras esterlinas. Es decir,
numericamente son 100 veces mas grandes.

- `Price` sigue siendo positivo.
- `Price` sigue siendo un numero float.
- No hay nulos.
- El tipo de dato no cambio.

Es un cambio **sistematico**, no un error. Y es exactamente lo que una regla
`Price > 0` NO detecta.

In [3]:
# 3. Simulamos el lote con el cambio silencioso de unidades.
lote_nuevo = X_te.copy()
lote_nuevo["Price"] = lote_nuevo["Price"] * 100  # libras -> peniques

y_real_nuevo = lote_nuevo["Price"] * lote_nuevo["Quantity"]  # el Revenue real tambien escala

# El pipeline predice sin lanzar errores:
preds_nuevo = pipe.predict(lote_nuevo)
mae_nuevo = mean_absolute_error(y_real_nuevo, preds_nuevo)

print(f"MAE sobre el lote corrupto: £{mae_nuevo:,.2f}")
print(f"MAE del lote limpio:         £{mae_base:,.2f}")
print(f"\n¿Lanzo alguna excepcion? NO.")
print(f"¿Hay nulos nuevos?          {lote_nuevo.isnull().any().any()}")
print(f"¿Cambiaron los tipos?       {dict(lote_nuevo.dtypes) == dict(X_te.dtypes)}")

MAE sobre el lote corrupto: £2,501.95
MAE del lote limpio:         £12.21

¿Lanzo alguna excepcion? NO.
¿Hay nulos nuevos?          False
¿Cambiaron los tipos?       True


El MAE se multiplico por un factor enorme. El modelo esta sirviendo
predicciones completamente incorrectas, pero no hay NINGUNA señal
automatica: ni excepcion, ni cambio de tipo, ni nulos.

Si este fuera un pipeline en produccion, el problema solo se detectaria
cuando alguien, por casualidad, revisara una metrica de negocio y notara
la discrepancia. Dias o semanas despues.

---

## El contrato de datos

Vamos a escribir un contrato con los tres niveles de check y vamos a ver
cual **si** atrapa el problema.

In [4]:
# 4. Contrato con tres niveles.
import pandera.pandas as pa

esquema = pa.DataFrameSchema(
    columns={
        # NIVEL 1 - por fila
        "Category": pa.Column(str,   checks=pa.Check.isin(["A", "B", "C", "D"])),
        "Country":  pa.Column(str,   checks=pa.Check.isin(["UK", "DE", "FR", "ES"])),
        "Price":    pa.Column(float, checks=[
            pa.Check.gt(0),                                                  # nivel 1
            pa.Check(lambda s: (s > 500).mean() < 0.01,                      # NIVEL 2 - por distribucion
                     error="mas del 1% de precios > 500: revisa unidades"),
        ]),
        "Quantity": pa.Column(int, checks=pa.Check.gt(0)),                   # nivel 1
    },
    checks=[
        # NIVEL 2 - por distribucion
        pa.Check(lambda df: df["Price"].mean() < 50,
                 error="precio medio demasiado alto: posible cambio de unidades"),
    ],
    strict=False,  # permite columnas extra como Revenue
)

# Primero verificamos que los datos limpios SI pasan el contrato.
esquema.validate(X_te)
print("Datos limpios: contrato pasa.")

Datos limpios: contrato pasa.


In [5]:
# 5. Ahora validamos el lote corrupto. Deberia fallar.
try:
    esquema.validate(lote_nuevo)
    print("FALLO: el contrato NO atrapo el cambio de unidades.")
except pa.errors.SchemaError as e:
    print("El contrato atrapo el cambio de unidades:\n")
    print(f"  {e}")

El contrato atrapo el cambio de unidades:

  Column 'Price' failed series or dataframe validator 1: <Check <lambda>: mas del 1% de precios > 500: revisa unidades>


## Lo importante

**La regla `Price > 0` (nivel 1) NO basto.** Cada precio individual, aun
multiplicado por 100, sigue siendo positivo. La fila a fila nunca atrapa
cambios sistematicos.

**La regla sobre la fraccion de precios > 500 (nivel 2) y la del precio medio
(nivel 2) son las que atrapan el problema.** Son reglas que miran el lote
completo y comparan contra lo que sabemos del dato historico.

Esta es la diferencia que decide si un pipeline sobrevive en produccion:

- Si tu contrato solo tiene nivel 1, estas desprotegido contra el 80% de los
  incidentes reales (cambios de unidad, degradacion del proveedor, cambios de
  volumen).
- Si tu contrato tiene al menos una regla de nivel 2 por cada variable critica,
  esta clase de incidentes falla el contrato antes de llegar al modelo.

---

## Siguiente paso

Lee [`contratos-de-datos.md`](contratos-de-datos.md) para el detalle de los
tres niveles, incluyendo el nivel 3 (entre columnas) que no cupo en este
notebook. Luego aplica la disciplina a **tu** dataset en el
[`taller.md`](taller.md).